<a href="https://colab.research.google.com/github/noorrami/ptbxl-ecg-classification/blob/main/notebooks/04_full_training_normalization_%26feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Phase 15: Environment Setup and Dataset Contract

This notebook establishes a reproducible, leakage-controlled normalization pipeline for the PTB-XL binary ECG classification study.

**Objectives**
- Load the previously audited training and validation waveform chunks.
- Preserve the official patient-disjoint data partitions.
- Estimate global per-lead normalization statistics exclusively from training recordings.
- Save reusable normalization parameters for subsequent machine learning experiments.
- Verify numerical correctness and data integrity without accessing held-out test waveforms.

**Dataset:** PTB-XL v1.0.3  
**Sampling frequency:** 100 Hz  
**Recording duration:** 10 seconds  
**Signal representation:** 1,000 time samples × 12 ECG leads  
**Training records:** 9,286  
**Validation records:** 1,147  

The held-out test set is excluded from all normalization fitting and validation procedures.

In [1]:
%pip install -q numpy pandas tqdm

In [2]:
from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/ML_Master_Project")

PROCESSED = ROOT / "data" / "processed"
CHUNKS = PROCESSED / "ecg_chunks"

RESULTS = ROOT / "results" / "normalization"
ARTIFACTS = ROOT / "artifacts" / "normalization"

RESULTS.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)

FS = 100
N_SAMPLES = 1000
N_LEADS = 12
CHUNK_SIZE = 64

EXPECTED_TRAIN_RECORDS = 9286
EXPECTED_VAL_RECORDS = 1147

EXPECTED_CONFIG_ID = "c37b7aee98472c39"

LEADS = [
    "I", "II", "III", "aVR", "aVL", "aVF",
    "V1", "V2", "V3", "V4", "V5", "V6"
]

train_df = pd.read_csv(
    PROCESSED / "train_manifest.csv",
    index_col="ecg_id"
).sort_index()

val_df = pd.read_csv(
    PROCESSED / "validation_manifest.csv",
    index_col="ecg_id"
).sort_index()

assert len(train_df) == EXPECTED_TRAIN_RECORDS
assert len(val_df) == EXPECTED_VAL_RECORDS

assert train_df.index.is_unique
assert val_df.index.is_unique

assert set(train_df.index).isdisjoint(set(val_df.index))

assert set(train_df["patient_id"]).isdisjoint(
    set(val_df["patient_id"])
)

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Configuration ID:", EXPECTED_CONFIG_ID)
print("PASS: Dataset contract validated.")

Mounted at /content/drive
Training records: 9286
Validation records: 1147
Configuration ID: c37b7aee98472c39
PASS: Dataset contract validated.


### Phase 16: Integrity-Aware Streaming Estimation

Global per-lead normalization parameters are estimated exclusively from the training partition.

For each ECG lead $c$, the population mean and standard deviation are defined as:

$$
\mu_c = \frac{1}{NT}\sum_{i=1}^{N}\sum_{t=1}^{T}x_{i,t,c}
$$

$$
\sigma_c =
\sqrt{
\frac{1}{NT}
\sum_{i=1}^{N}\sum_{t=1}^{T}
(x_{i,t,c}-\mu_c)^2
}
$$

where $N$ is the number of training recordings, $T=1000$ is the number of samples per recording, and $c$ identifies one of the twelve ECG leads.

To improve numerical stability and memory efficiency, chunk-level means and second central moments are merged using a parallel variance aggregation procedure.

All statistics are computed in `float64`. The held-out test partition is not accessed.

In [3]:
def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            block = file.read(block_size)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()


def read_verified_chunk(split_name, chunk_index, manifest):
    start = chunk_index * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(manifest))

    expected = manifest.iloc[start:end]

    folder = CHUNKS / split_name
    path = folder / f"chunk_{chunk_index:04d}.npz"
    checksum_path = folder / f"chunk_{chunk_index:04d}.sha256"

    if not path.exists():
        raise FileNotFoundError(path)

    if not checksum_path.exists():
        raise FileNotFoundError(checksum_path)

    expected_hash = checksum_path.read_text().strip()
    actual_hash = sha256_file(path)

    if expected_hash != actual_hash:
        raise RuntimeError(
            f"SHA-256 mismatch: {split_name}/{path.name}"
        )

    with np.load(path, allow_pickle=False) as data:
        X = data["X"].copy()
        y = data["y"].copy()
        ecg_ids = data["ecg_ids"].copy()
        patient_ids = data["patient_ids"].copy()
        config_id = str(data["config_id"].item())

    if config_id != EXPECTED_CONFIG_ID:
        raise RuntimeError("Acquisition configuration mismatch.")

    if X.shape != (len(expected), N_SAMPLES, N_LEADS):
        raise RuntimeError(
            f"Invalid waveform shape: {X.shape}"
        )

    if X.dtype != np.float32:
        raise RuntimeError("Unexpected waveform dtype.")

    if not np.isfinite(X).all():
        raise RuntimeError("Non-finite waveform values detected.")

    np.testing.assert_array_equal(
        ecg_ids.astype(np.int64),
        expected.index.to_numpy(dtype=np.int64)
    )

    np.testing.assert_array_equal(
        patient_ids.astype(np.int64),
        expected["patient_id"].to_numpy(dtype=np.int64)
    )

    np.testing.assert_array_equal(
        y.astype(np.int64),
        expected["target"].to_numpy(dtype=np.int64)
    )

    return X, y

In [4]:
def compute_training_statistics():
    count = 0

    mean = np.zeros(N_LEADS, dtype=np.float64)
    M2 = np.zeros(N_LEADS, dtype=np.float64)

    records_seen = 0

    n_chunks = (
        len(train_df) + CHUNK_SIZE - 1
    ) // CHUNK_SIZE

    for chunk_index in tqdm(
        range(n_chunks),
        desc="Estimating train-only scaler"
    ):
        X, _ = read_verified_chunk(
            "train",
            chunk_index,
            train_df
        )

        X64 = X.astype(np.float64)

        batch_count = X64.shape[0] * X64.shape[1]
        batch_mean = X64.mean(axis=(0, 1))

        batch_M2 = np.square(
            X64 - batch_mean
        ).sum(axis=(0, 1))

        if count == 0:
            mean = batch_mean
            M2 = batch_M2
            count = batch_count
        else:
            delta = batch_mean - mean
            new_count = count + batch_count

            mean = mean + delta * (
                batch_count / new_count
            )

            M2 = (
                M2
                + batch_M2
                + np.square(delta)
                * count * batch_count / new_count
            )

            count = new_count

        records_seen += X.shape[0]

    assert records_seen == EXPECTED_TRAIN_RECORDS
    assert count == EXPECTED_TRAIN_RECORDS * N_SAMPLES

    variance = M2 / count
    std = np.sqrt(variance)

    assert np.isfinite(mean).all()
    assert np.isfinite(std).all()
    assert np.all(std > 1e-12)

    return mean, std, records_seen, count


mean, std, n_records, n_samples = (
    compute_training_statistics()
)

statistics_df = pd.DataFrame({
    "lead": LEADS,
    "mean": mean,
    "std": std
})

print(statistics_df.to_string(index=False))

print("\nTraining records:", n_records)
print("Samples per lead:", n_samples)
print("PASS: Training-only statistics computed.")

Estimating train-only scaler:   0%|          | 0/146 [00:00<?, ?it/s]

lead      mean      std
   I -0.002150 0.149343
  II -0.001752 0.152185
 III  0.000398 0.136296
 aVR  0.001945 0.134511
 aVL -0.001206 0.121017
 aVF -0.000638 0.123527
  V1 -0.000062 0.185883
  V2 -0.001041 0.284965
  V3 -0.001720 0.274077
  V4 -0.001361 0.258652
  V5 -0.000951 0.239443
  V6 -0.003173 0.290475

Training records: 9286
Samples per lead: 9286000
PASS: Training-only statistics computed.


### Phase 17: Scaler Serialization and Provenance

The global per-lead normalization parameters are persisted as a reusable experimental artifact.

The scaler is fitted exclusively on the official training partition and is associated with the verified acquisition configuration, training manifest, and dataset version.

For each ECG lead $c$, subsequent transformations will use:

$$
z_{i,t,c} = \frac{x_{i,t,c} - \mu_c}{\sigma_c}
$$

The same frozen parameters will be applied to training, validation, and eventually held-out test recordings. No statistics from validation or test recordings will be used to refit the scaler.

In [5]:
import os
from datetime import datetime, timezone

SCALER_PATH = ARTIFACTS / "train_lead_zscore.npz"
METADATA_PATH = ARTIFACTS / "train_lead_zscore_metadata.json"
STATS_CSV_PATH = RESULTS / "training_lead_statistics.csv"

manifest_path = PROCESSED / "train_manifest.csv"
manifest_hash = sha256_file(manifest_path)

scaler_metadata = {
    "dataset": "PTB-XL",
    "dataset_version": "1.0.3",
    "method": "global_per_lead_zscore",
    "fitted_on": "train_only",
    "training_records": int(n_records),
    "samples_per_lead": int(n_samples),
    "sampling_frequency_hz": FS,
    "signal_shape": [N_SAMPLES, N_LEADS],
    "lead_order": LEADS,
    "acquisition_config_id": EXPECTED_CONFIG_ID,
    "train_manifest_sha256": manifest_hash,
    "variance_definition": "population",
    "calculation_dtype": "float64"
}

if SCALER_PATH.exists():
    with np.load(SCALER_PATH, allow_pickle=False) as saved:
        np.testing.assert_allclose(
            saved["mean"], mean, rtol=1e-12, atol=1e-12
        )
        np.testing.assert_allclose(
            saved["std"], std, rtol=1e-12, atol=1e-12
        )
        assert str(saved["config_id"].item()) == EXPECTED_CONFIG_ID

    print("Existing scaler verified.")

else:
    temporary_path = SCALER_PATH.with_name(
        SCALER_PATH.stem + ".tmp.npz"
    )

    np.savez_compressed(
        temporary_path,
        mean=mean.astype(np.float64),
        std=std.astype(np.float64),
        config_id=np.array(EXPECTED_CONFIG_ID),
        train_manifest_sha256=np.array(manifest_hash),
        n_records=np.array(n_records),
        n_samples=np.array(n_samples)
    )

    os.replace(temporary_path, SCALER_PATH)
    print("Scaler saved:", SCALER_PATH)

if METADATA_PATH.exists():
    previous_metadata = json.loads(
        METADATA_PATH.read_text(encoding="utf-8")
    )

    for key, value in scaler_metadata.items():
        assert previous_metadata[key] == value, (
            f"Metadata mismatch: {key}"
        )
else:
    scaler_metadata["created_at_utc"] = (
        datetime.now(timezone.utc).isoformat()
    )

    METADATA_PATH.write_text(
        json.dumps(
            scaler_metadata,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

statistics_df.to_csv(STATS_CSV_PATH, index=False)

print("Scaler SHA-256:", sha256_file(SCALER_PATH))
print("Metadata:", METADATA_PATH)
print("Statistics:", STATS_CSV_PATH)
print("PASS: Scaler artifact and provenance recorded.")

Scaler saved: /content/drive/MyDrive/ML_Master_Project/artifacts/normalization/train_lead_zscore.npz
Scaler SHA-256: ca01d553a32ba6c0960a5f85289b27a82197ecf507b887afada24e208e2572f4
Metadata: /content/drive/MyDrive/ML_Master_Project/artifacts/normalization/train_lead_zscore_metadata.json
Statistics: /content/drive/MyDrive/ML_Master_Project/results/normalization/training_lead_statistics.csv
PASS: Scaler artifact and provenance recorded.


### Phase 18: Full-Training Numerical Validation

The saved normalization parameters are reloaded and applied to all training recordings in a streaming manner.

For every ECG lead, the normalized training distribution should satisfy:

$$
\mathbb{E}[Z_c] \approx 0
$$

$$
\operatorname{Std}(Z_c) \approx 1
$$

These conditions are evaluated across the complete training partition, rather than on individual chunks.

This verification confirms numerical consistency of the fitted scaler. It does not imply that ECG signals follow a Gaussian distribution.

In [7]:
with np.load(SCALER_PATH, allow_pickle=False) as saved:
    saved_mean = saved["mean"].astype(np.float64)
    saved_std = saved["std"].astype(np.float64)

np.testing.assert_allclose(saved_mean, mean)
np.testing.assert_allclose(saved_std, std)

normalized_sum = np.zeros(N_LEADS, dtype=np.float64)
normalized_sum_sq = np.zeros(N_LEADS, dtype=np.float64)
normalized_count = 0

n_train_chunks = (
    len(train_df) + CHUNK_SIZE - 1
) // CHUNK_SIZE

for chunk_index in tqdm(
    range(n_train_chunks),
    desc="Validating normalized train"
):
    X, _ = read_verified_chunk(
        "train", chunk_index, train_df
    )

    Z = (
        X.astype(np.float64) - saved_mean
    ) / saved_std

    assert np.isfinite(Z).all()

    normalized_sum += Z.sum(axis=(0, 1))
    normalized_sum_sq += np.square(Z).sum(axis=(0, 1))
    normalized_count += Z.shape[0] * Z.shape[1]

normalized_mean = normalized_sum / normalized_count

normalized_variance = (
    normalized_sum_sq / normalized_count
    - np.square(normalized_mean)
)

normalized_std = np.sqrt(
    np.maximum(normalized_variance, 0.0)
)

validation_df = pd.DataFrame({
    "lead": LEADS,
    "normalized_mean": normalized_mean,
    "normalized_std": normalized_std
})

print(validation_df.to_string(index=False))

np.testing.assert_allclose(
    normalized_mean,
    np.zeros(N_LEADS),
    atol=1e-6
)

np.testing.assert_allclose(
    normalized_std,
    np.ones(N_LEADS),
    atol=1e-6
)

print("\nPASS: Full-training normalization verified.")

Validating normalized train:   0%|          | 0/146 [00:00<?, ?it/s]

lead  normalized_mean  normalized_std
   I     4.602474e-16             1.0
  II     5.474071e-17             1.0
 III     1.146810e-15             1.0
 aVR    -4.689965e-16             1.0
 aVL     8.088036e-16             1.0
 aVF    -1.615088e-16             1.0
  V1    -7.048575e-17             1.0
  V2    -9.548253e-18             1.0
  V3    -1.209476e-17             1.0
  V4     7.484190e-18             1.0
  V5     1.586364e-17             1.0
  V6    -6.247126e-16             1.0

PASS: Full-training normalization verified.


### Phase 19: Validation Distribution Analysis

The frozen training-derived scaler is applied to the validation partition without refitting.

Unlike the normalized training partition, validation recordings are not expected to exhibit exactly zero mean or unit standard deviation.

Differences between the two partitions may reflect sampling variability, patient characteristics, signal amplitude differences, or distributional shifts.

This analysis is descriptive and does not establish clinical generalizability or quantify model performance.

In [8]:
val_sum = np.zeros(N_LEADS, dtype=np.float64)
val_sum_sq = np.zeros(N_LEADS, dtype=np.float64)
val_count = 0

n_val_chunks = (
    len(val_df) + CHUNK_SIZE - 1
) // CHUNK_SIZE

for chunk_index in tqdm(
    range(n_val_chunks),
    desc="Analyzing validation distribution"
):
    X, _ = read_verified_chunk(
        "validation", chunk_index, val_df
    )

    Z = (
        X.astype(np.float64) - saved_mean
    ) / saved_std

    assert np.isfinite(Z).all()

    val_sum += Z.sum(axis=(0, 1))
    val_sum_sq += np.square(Z).sum(axis=(0, 1))
    val_count += Z.shape[0] * Z.shape[1]

val_mean = val_sum / val_count

val_variance = (
    val_sum_sq / val_count
    - np.square(val_mean)
)

val_std = np.sqrt(
    np.maximum(val_variance, 0.0)
)

distribution_df = pd.DataFrame({
    "lead": LEADS,
    "train_mean": normalized_mean,
    "train_std": normalized_std,
    "validation_mean": val_mean,
    "validation_std": val_std
})

distribution_df.to_csv(
    RESULTS / "train_validation_distribution.csv",
    index=False
)

print(distribution_df.round(5).to_string(index=False))

print("\nPASS: Validation analyzed using frozen training scaler.")

Analyzing validation distribution:   0%|          | 0/18 [00:00<?, ?it/s]

lead  train_mean  train_std  validation_mean  validation_std
   I         0.0        1.0          0.00689         0.90834
  II         0.0        1.0          0.00584         1.04044
 III         0.0        1.0         -0.00104         0.93016
 aVR        -0.0        1.0         -0.00711         0.98979
 aVL         0.0        1.0          0.00477         0.86532
 aVF        -0.0        1.0          0.00298         1.02189
  V1        -0.0        1.0          0.00106         0.99465
  V2        -0.0        1.0          0.00038         0.98829
  V3        -0.0        1.0          0.00073         1.02004
  V4         0.0        1.0          0.00075         0.99247
  V5         0.0        1.0          0.00133         0.99009
  V6        -0.0        1.0          0.00857         0.71821

PASS: Validation analyzed using frozen training scaler.


### Phase 20: ECG Feature Engineering

This phase constructs a reproducible statistical feature representation of twelve-lead ECG recordings for conventional supervised learning models.

The initial feature set includes per-lead amplitude statistics, root mean square (RMS), signal energy, and first-difference descriptors.

Features are extracted from the original waveform amplitudes. A separate feature-level scaler will subsequently be fitted exclusively on the training partition.

These descriptors characterize signal properties but are not assumed to correspond directly to clinically validated ECG intervals or diagnostic biomarkers.

In [9]:
import numpy as np
import pandas as pd

LEADS = [
    "I", "II", "III", "aVR", "aVL", "aVF",
    "V1", "V2", "V3", "V4", "V5", "V6"
]

FEATURE_NAMES = [
    "mean",
    "std",
    "min",
    "max",
    "peak_to_peak",
    "rms",
    "mean_abs",
    "mean_abs_diff",
    "std_diff",
    "energy"
]

def extract_ecg_features(X):
    """
    X: float array of shape (N, 1000, 12)

    Returns:
        DataFrame with N rows and 120 features.
    """
    X = np.asarray(X, dtype=np.float64)

    if X.ndim != 3 or X.shape[1:] != (1000, 12):
        raise ValueError(f"Unexpected ECG shape: {X.shape}")

    if not np.isfinite(X).all():
        raise ValueError("Non-finite ECG values detected.")

    D = np.diff(X, axis=1)

    features = {}

    for lead_idx, lead in enumerate(LEADS):
        S = X[:, :, lead_idx]
        SD = D[:, :, lead_idx]

        features[f"{lead}_mean"] = S.mean(axis=1)
        features[f"{lead}_std"] = S.std(axis=1)
        features[f"{lead}_min"] = S.min(axis=1)
        features[f"{lead}_max"] = S.max(axis=1)

        features[f"{lead}_peak_to_peak"] = (
            S.max(axis=1) - S.min(axis=1)
        )

        features[f"{lead}_rms"] = np.sqrt(
            np.mean(S ** 2, axis=1)
        )

        features[f"{lead}_mean_abs"] = (
            np.mean(np.abs(S), axis=1)
        )

        features[f"{lead}_mean_abs_diff"] = (
            np.mean(np.abs(SD), axis=1)
        )

        features[f"{lead}_std_diff"] = SD.std(axis=1)

        features[f"{lead}_energy"] = (
            np.sum(S ** 2, axis=1)
        )

    result = pd.DataFrame(features)

    if not np.isfinite(result.to_numpy()).all():
        raise RuntimeError("Invalid extracted features.")

    return result

### Phase 21: Pilot Feature Extraction and Quality Checks

The feature extraction procedure is first evaluated on a verified training chunk.

The pilot test checks dimensional consistency, feature naming, finite numerical values, and compatibility with the existing ECG acquisition pipeline.

No model fitting or held-out test evaluation is performed in this phase.

In [10]:
X_pilot, y_pilot = read_verified_chunk(
    "train",
    0,
    train_df
)

pilot_features = extract_ecg_features(X_pilot)

assert pilot_features.shape == (64, 120)
assert pilot_features.columns.is_unique
assert not pilot_features.isna().any().any()
assert np.isfinite(pilot_features.to_numpy()).all()

print("Input ECG shape:", X_pilot.shape)
print("Extracted features:", pilot_features.shape)
print("Number of labels:", len(y_pilot))

print("\nFirst 12 feature names:")
print(pilot_features.columns[:12].tolist())

print("\nFeature preview:")
display(pilot_features.iloc[:5, :8].round(5))

print("\nPASS: Pilot ECG feature extraction verified.")

Input ECG shape: (64, 1000, 12)
Extracted features: (64, 120)
Number of labels: 64

First 12 feature names:
['I_mean', 'I_std', 'I_min', 'I_max', 'I_peak_to_peak', 'I_rms', 'I_mean_abs', 'I_mean_abs_diff', 'I_std_diff', 'I_energy', 'II_mean', 'II_std']

Feature preview:


,I_mean,I_std,I_min,I_max,I_peak_to_peak,I_rms,I_mean_abs,I_mean_abs_diff
0,0.00151,0.10902,-0.195,0.706,0.901,0.10903,0.06902,0.03364
1,-0.00218,0.13343,-0.349,0.664,1.013,0.13345,0.09148,0.02750
2,-0.00277,0.12120,-0.164,0.827,0.991,0.12123,0.07382,0.03518
3,-0.00326,0.13780,-0.576,0.503,1.079,0.13784,0.08895,0.04086
4,-0.00377,0.08271,-0.267,0.440,0.707,0.08279,0.05509,0.02571



PASS: Pilot ECG feature extraction verified.


### Phase 22: Full-Dataset Feature Extraction

This phase applies the verified ECG feature extractor to the complete training and validation partitions.

Each ECG recording is represented by 120 statistical and temporal features derived from its original twelve-lead waveform.

The extraction procedure is deterministic and label-independent. ECG identifiers, patient identifiers, and class labels are retained to preserve sample alignment and enable patient-level integrity checks.

The held-out test waveforms remain untouched.

Feature standardization, feature selection, and model fitting will be performed in subsequent phases using training-derived parameters only.

In [11]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

FEATURE_DIR = PROCESSED / "engineered_features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_RESULTS = ROOT / "results" / "feature_engineering"
FEATURE_RESULTS.mkdir(parents=True, exist_ok=True)

EXPECTED_FEATURE_COUNT = 120

EXPECTED_FEATURE_COLUMNS = [
    f"{lead}_{feature}"
    for lead in LEADS
    for feature in FEATURE_NAMES
]

assert len(EXPECTED_FEATURE_COLUMNS) == EXPECTED_FEATURE_COUNT

print("Feature directory:", FEATURE_DIR)
print("Feature count:", len(EXPECTED_FEATURE_COLUMNS))
print("PASS: Feature extraction configuration ready.")

Feature directory: /content/drive/MyDrive/ML_Master_Project/data/processed/engineered_features
Feature count: 120
PASS: Feature extraction configuration ready.


In [12]:
def extract_split_features(split_name, manifest):
    """
    Extract engineered features from all verified ECG chunks.

    Returns:
        DataFrame indexed by ecg_id with:
        patient_id, target, and 120 engineered features.
    """

    if split_name not in {"train", "validation"}:
        raise ValueError("Only train and validation are permitted.")

    n_chunks = (
        len(manifest) + CHUNK_SIZE - 1
    ) // CHUNK_SIZE

    frames = []

    for chunk_index in tqdm(
        range(n_chunks),
        desc=f"Extracting {split_name} features"
    ):
        X, y = read_verified_chunk(
            split_name,
            chunk_index,
            manifest
        )

        start = chunk_index * CHUNK_SIZE
        end = min(start + CHUNK_SIZE, len(manifest))

        expected = manifest.iloc[start:end]

        features = extract_ecg_features(X)

        if features.shape != (
            len(expected), EXPECTED_FEATURE_COUNT
        ):
            raise RuntimeError(
                f"Invalid feature dimensions in chunk {chunk_index}"
            )

        if features.columns.tolist() != EXPECTED_FEATURE_COLUMNS:
            raise RuntimeError(
                f"Feature order mismatch in chunk {chunk_index}"
            )

        np.testing.assert_array_equal(
            y.astype(np.int64),
            expected["target"].to_numpy(dtype=np.int64)
        )

        features.index = expected.index.copy()
        features.index.name = "ecg_id"

        features.insert(
            0,
            "patient_id",
            expected["patient_id"].to_numpy()
        )

        features.insert(
            1,
            "target",
            y.astype(np.int64)
        )

        frames.append(features)

    result = pd.concat(frames, axis=0)

    if len(result) != len(manifest):
        raise RuntimeError("Record count mismatch.")

    if not result.index.equals(manifest.index):
        raise RuntimeError("ECG identifier alignment mismatch.")

    if not result.index.is_unique:
        raise RuntimeError("Duplicate ECG identifiers detected.")

    if result.isna().any().any():
        raise RuntimeError("Missing feature values detected.")

    feature_matrix = result[EXPECTED_FEATURE_COLUMNS].to_numpy()

    if not np.isfinite(feature_matrix).all():
        raise RuntimeError("Non-finite features detected.")

    return result

In [13]:
train_features_path = FEATURE_DIR / "train_features.csv"
val_features_path = FEATURE_DIR / "validation_features.csv"

train_features = extract_split_features(
    "train",
    train_df
)

val_features = extract_split_features(
    "validation",
    val_df
)

assert len(train_features) == 9286
assert len(val_features) == 1147

assert set(train_features["patient_id"]).isdisjoint(
    set(val_features["patient_id"])
)

assert train_features.columns.equals(val_features.columns)

def save_features_atomic(frame, path):
    temporary_path = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary_path, index=True, float_format="%.17g")
    os.replace(temporary_path, path)

save_features_atomic(train_features, train_features_path)
save_features_atomic(val_features, val_features_path)

print("Train feature matrix:", train_features.shape)
print("Validation feature matrix:", val_features.shape)

print("Training classes:")
print(train_features["target"].value_counts().sort_index())

print("\nValidation classes:")
print(val_features["target"].value_counts().sort_index())

print("\nSaved:")
print(train_features_path)
print(val_features_path)

print("\nPASS: Full-dataset feature extraction completed.")

Extracting train features:   0%|          | 0/146 [00:00<?, ?it/s]

Extracting validation features:   0%|          | 0/18 [00:00<?, ?it/s]

Train feature matrix: (9286, 122)
Validation feature matrix: (1147, 122)
Training classes:
target
0    7243
1    2043
Name: count, dtype: int64

Validation classes:
target
0    914
1    233
Name: count, dtype: int64

Saved:
/content/drive/MyDrive/ML_Master_Project/data/processed/engineered_features/train_features.csv
/content/drive/MyDrive/ML_Master_Project/data/processed/engineered_features/validation_features.csv

PASS: Full-dataset feature extraction completed.


### Phase 23: Engineered Feature Quality Audit

The complete feature matrices are audited for missing values, non-finite values, constant features, and strong pairwise correlations.

Feature redundancy is examined using training data only. No feature selection decisions are made using held-out test data.

The purpose of this phase is to characterize the engineered representation and identify potential numerical or statistical issues before supervised model development.

In [14]:
X_train_features = train_features[EXPECTED_FEATURE_COLUMNS]

feature_variances = X_train_features.var(ddof=0)

constant_features = feature_variances[
    feature_variances <= 1e-12
].index.tolist()

correlation_matrix = X_train_features.corr().abs()

upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape, dtype=bool),
        k=1
    )
)

high_corr_pairs = (
    upper_triangle.stack()
    .loc[lambda values: values >= 0.98]
    .sort_values(ascending=False)
)

print("Total features:", X_train_features.shape[1])
print("Constant / near-constant features:", len(constant_features))
print("Highly correlated pairs (|r| >= 0.98):", len(high_corr_pairs))

print("\nConstant features:")
print(constant_features)

print("\nTop 15 highly correlated pairs:")
print(high_corr_pairs.head(15).to_string())

audit_summary = {
    "train_records": int(len(train_features)),
    "validation_records": int(len(val_features)),
    "feature_count": EXPECTED_FEATURE_COUNT,
    "constant_features": constant_features,
    "high_correlation_pair_count": int(len(high_corr_pairs)),
    "correlation_threshold": 0.98,
    "test_waveforms_accessed": False
}

audit_path = FEATURE_RESULTS / "feature_quality_audit.json"

audit_path.write_text(
    json.dumps(audit_summary, indent=2),
    encoding="utf-8"
)

print("\nPASS: Feature quality audit completed.")

Total features: 120
Constant / near-constant features: 0
Highly correlated pairs (|r| >= 0.98): 10

Constant features:
[]

Top 15 highly correlated pairs:
V6_mean     V6_energy      0.999254
I_energy    aVL_energy     0.998865
V5_std      V5_rms         0.998718
V3_std      V3_rms         0.998329
III_energy  aVL_energy     0.997999
V4_std      V4_rms         0.997566
V2_std      V2_rms         0.996409
I_energy    III_energy     0.995315
V6_rms      V6_mean_abs    0.989528
V1_std      V1_rms         0.988183

PASS: Feature quality audit completed.
